# Parquet Benchmarks – Backtesting Experiments

## Libraries

In [ ]:
from fabric import ThreadingGroup, Connection
from chi import server, ssh
import os
import warnings
import time
import sys
sys.path.append('../scripts')
from utilities import *
warnings.filterwarnings('ignore')

## Configurations

In [ ]:
sshUser = "ubuntu"
client_ip = "64.131.114.143"

storageType = {"HDD": "/HDD", "NVME": "/NVME_RAID-0"}
WORKING_DIR = storageType["NVME"]

targetClients = [1, 2, 4, 8, 16, 32, 64]

## Initialize environment

In [ ]:
# These groups are for processing in parallel 
clientConnection = makeConnection(client_ip, user=sshUser)

# Update & Upgrade latest packages
clientConnection.run("sudo apt update && sudo apt upgrade -y", hide=True)
rebootGroup(clientConnection)

Wait for all server to come online after reboot

In [ ]:
# Buffer of reboot (no more +1 min shutdown-schedule delay)
time.sleep(240)
waitForSSH(client_ip, user=sshUser, timeout=1200)

## Benchmark server

Prepare benchmark server

In [ ]:
clientConnection.run(f"sudo chmod -R 777 {WORKING_DIR}", hide=True);

## Benchmark Client (Backtester)

In [ ]:
clientConnection.put("../scripts/backtesterSkipping.py")
clientConnection.put("../scripts/initBacktestClient.sh")
clientConnection.run("bash initBacktestClient.sh", hide=True);

## Bootstrap Database

In [ ]:
clientConnection.put("../scripts/restoreDB.sh")

# Restore DB
clientConnection.run(f"bash restoreDB.sh MYSTIC@IIT CSV BACKTESTER {WORKING_DIR}", hide=True)

# Keep checking if job is done running in tmux window
checkTmuxJobIsDone(clientConnection);

## Experiments

**Perform experiments**

In [ ]:
runWorkload("BACKTEST", targetClients, "127.0.0.1", 0000, "CSV", [client_ip], 100, 10, clientConnection, WORKING_DIR, "ubuntu", 1000000, 900, f"{WORKING_DIR}/BacktestData.csv", sleepInterval=60)